In [ ]:
import os
import pandas as pd

# 염색체 목록
chromosomes = [str(i) for i in range(1, 23)] + ['X', 'Y', 'M']

# 경로 설정
path_51_dir = r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\annotateAll"
path_52_dir = r"E:\CAGI_data\dbNSFP5.1a_grch38_splits"
output_dir   = r"E:\CAGI_data\All_Missense_Annotated"
os.makedirs(output_dir, exist_ok=True)

# 불필요한 열
cols_to_drop_51 = [
    "hg19_chr", "hg19_pos(1-based)", "hg18_chr", "hg18_pos(1-based)",
    "genename", "cds_strand", "refcodon", "codonpos",
    "Ensembl_geneid", "Ensembl_transcriptid", "Ensembl_proteinid"
]

# 병합 키 생성 함수
def make_key(df, chr_col):
    return df[chr_col].astype(str) + "__" + \
           df["pos(1-based)"].astype(str) + "__" + \
           df["ref"].astype(str) + "__" + \
           df["alt"].astype(str) + "__" + \
           df["aaref"].astype(str) + "__" + \
           df["aaalt"].astype(str)

# 염색체별 처리
for chrom in chromosomes:
    print(f"🔄 Processing chr{chrom}...")

    # 1. 파일 경로
    file_51 = os.path.join(path_51_dir, f"dbNSFP5.1_nsSNV.chr{chrom}.gz")
    file_52 = os.path.join(path_52_dir, f"dbNSFP5.2a_{chrom}.tsv.gz")
    
    # 2. 데이터 불러오기
    df_51 = pd.read_csv(file_51, sep='\t', compression='gzip', low_memory=False, dtype=str)
    df_52 = pd.read_csv(file_52, sep='\t', compression='gzip', low_memory=False, dtype=str)

    # 3. 병합 키 생성
    df_51["__key__"] = make_key(df_51, "#chr")
    df_52["__key__"] = make_key(df_52, "chr")

    # 4. 5.2a에서 필요한 열만, aapos → mut_pos로 변경
    df_52_slim = df_52[["__key__", "Uniprot_acc", "Uniprot_entry", "aapos"]].copy()
    df_52_slim = df_52_slim.rename(columns={"aapos": "mut_pos"})
    df_52_slim = df_52_slim.drop_duplicates("__key__")

    # 5. 병합
    df_merged = df_51.merge(df_52_slim, on="__key__", how="left")

    # 6. 누락/중복 로그
    dups = df_52[df_52.duplicated("__key__", keep=False)]
    if not dups.empty:
        dups.to_csv(os.path.join(output_dir, f"log_duplicated_keys_chr{chrom}.tsv"), sep="\t", index=False)
    
    missing = df_merged[df_merged["Uniprot_acc"].isna()]
    if not missing.empty:
        missing.to_csv(os.path.join(output_dir, f"log_missing_uniprot_chr{chrom}.tsv"), sep="\t", index=False)

    # 7. 열 제거 및 저장
    df_merged.drop(columns=["__key__"], inplace=True)
    for col in cols_to_drop_51:
        if col in df_merged.columns:
            df_merged.drop(columns=col, inplace=True)

    out_path = os.path.join(output_dir, f"chr{chrom}.tsv")
    df_merged.to_csv(out_path, sep="\t", index=False)

    print(f"✅ chr{chrom} done. Saved to {out_path}")

print("\n🎉 All chromosomes processed!")


In [ ]:
df_merged

In [ ]:
import os
import pandas as pd

input_dir = r"E:\CAGI_data\All_Missense_Annotated"
output_tsv = r"E:\CAGI_data\All_Missense_Annotated\all_missense_variants_for_prediction.tsv"

dfs = []

for file in os.listdir(input_dir):

    print(file)
    
    if not file.endswith(".tsv") or file.startswith("log_"):
        continue

    path = os.path.join(input_dir, file)
    df = pd.read_csv(path, sep="\t", dtype=str)

    # 필수 컬럼 확인
    if not {"Uniprot_acc", "mut_pos", "aaref", "aaalt"}.issubset(df.columns):
        print(f"⚠️ Skipped {file}: required columns not found")
        continue

    # drop rows with missing split targets
    df = df.dropna(subset=["Uniprot_acc", "mut_pos", "aaref", "aaalt"])

    # split Uniprot_acc & mut_pos into exploded rows
    df["Uniprot_acc"] = df["Uniprot_acc"].str.split(";")
    df["mut_pos"] = df["mut_pos"].str.split(";")

    df = df.explode("Uniprot_acc").explode("mut_pos")

    df_clean = df[["Uniprot_acc", "mut_pos", "aaref", "aaalt"]].copy()
    df_clean.columns = ["UniProtID", "MutPos", "WT", "Mut"]

    dfs.append(df_clean)

# # 모든 조각 합치고 중복 제거
# df_out = pd.concat(dfs, ignore_index=True)
# df_out.drop_duplicates(inplace=True)

# # 저장
# df_out.to_csv(output_tsv, sep="\t", index=False)
# print(f"\n✅ Done! Output saved to:\n{output_tsv}")


In [ ]:
# 염색체 이름 순서 (dfs의 순서와 동일하다고 가정)
chroms = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY", "chrM"]
assert len(chroms) == len(dfs), "❗ dfs와 염색체 수 불일치!"

for df_chr, chrom in zip(dfs, chroms):
    out_path = os.path.join(input_dir, f"missense_for_prediction_{chrom}.tsv")
    df_chr.drop_duplicates(inplace=True)
    df_chr.to_csv(out_path, sep="\t", index=False)
    print(f"✅ Saved: {out_path}")

In [1]:
import os

input_dir = r"E:\CAGI_data\All_Missense_Annotated"
chroms = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY", "chrM"]

all_ids = set()

for chrom in chroms:
    path = os.path.join(input_dir, f"missense_for_prediction_{chrom}.tsv")
    if not os.path.exists(path):
        print(f"⚠️ Missing file: {path}")
        continue

    with open(path, "r") as f:
        header = f.readline().strip().split("\t")  # 첫 줄에서 컬럼명 읽기
        try:
            col_idx = header.index("UniProtID")    # UniProtID 컬럼 인덱스
        except ValueError:
            print(f"⚠️ UniProtID column not found in {path}")
            continue

        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) > col_idx:
                uid = parts[col_idx].strip()
                if uid:  # 빈 값 제외
                    all_ids.add(uid)

print(f"✅ 총 UniProtID 개수: {len(all_ids)}")

# 정렬해서 저장
out_path = os.path.join(input_dir, "unique_uniprot_ids.txt")
with open(out_path, "w") as f:
    for uid in sorted(all_ids):
        f.write(uid + "\n")

print(f"💾 Saved unique IDs to {out_path}")


✅ 총 UniProtID 개수: 70116
💾 Saved unique IDs to E:\CAGI_data\All_Missense_Annotated\unique_uniprot_ids.txt


In [6]:
sorted(all_ids)

['.',
 'A0A024QZ33',
 'A0A024QZ42',
 'A0A024QZP7',
 'A0A024QZW4',
 'A0A024QZX5',
 'A0A024R0Y4',
 'A0A024R161',
 'A0A024R1R8',
 'A0A024R214',
 'A0A024R3B8',
 'A0A024R3B9',
 'A0A024R3M2',
 'A0A024R3Z1',
 'A0A024R4K9',
 'A0A024R571',
 'A0A024R5G9',
 'A0A024R772',
 'A0A024R7P0',
 'A0A024R7W5',
 'A0A024R8F3',
 'A0A024RA87',
 'A0A024RBG1',
 'A0A024RBT8',
 'A0A024RCL3',
 'A0A024RCN4',
 'A0A024RDL5',
 'A0A044PY82',
 'A0A067XG54',
 'A0A067XG57',
 'A0A075B6E2',
 'A0A075B6E5',
 'A0A075B6E6',
 'A0A075B6E9',
 'A0A075B6F3',
 'A0A075B6F6',
 'A0A075B6F9',
 'A0A075B6G4',
 'A0A075B6G5',
 'A0A075B6G6',
 'A0A075B6G7',
 'A0A075B6G8',
 'A0A075B6H0',
 'A0A075B6H3',
 'A0A075B6H4',
 'A0A075B6P0',
 'A0A075B6P4',
 'A0A075B6P6',
 'A0A075B6P9',
 'A0A075B6Q6',
 'A0A075B6R4',
 'A0A075B6R5',
 'A0A075B6R8',
 'A0A075B6T1',
 'A0A075B6T9',
 'A0A075B723',
 'A0A075B724',
 'A0A075B727',
 'A0A075B728',
 'A0A075B729',
 'A0A075B730',
 'A0A075B731',
 'A0A075B734',
 'A0A075B735',
 'A0A075B736',
 'A0A075B737',
 'A0A075B740',
 'A0

In [4]:
import json
import os

# JSON 불러오기
json_path = r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\UniProtID_to_seq.json"
with open(json_path, "r") as f:
    id_to_seq = json.load(f)

# all_ids는 이미 확보된 상태라고 가정
print(f"총 UniProtID 개수: {len(all_ids)}")

# 매핑 안 된 ID 찾기
unmatched_ids = sorted([uid for uid in all_ids if uid not in id_to_seq])

print(f"❗ 매핑 안 된 ID 개수: {len(unmatched_ids)}")

총 UniProtID 개수: 70116
❗ 매핑 안 된 ID 개수: 51383


In [5]:
unmatched_ids

['.',
 'A0A024QZ33',
 'A0A024QZ42',
 'A0A024QZP7',
 'A0A024QZW4',
 'A0A024QZX5',
 'A0A024R0Y4',
 'A0A024R161',
 'A0A024R214',
 'A0A024R3B8',
 'A0A024R3B9',
 'A0A024R3M2',
 'A0A024R3Z1',
 'A0A024R4K9',
 'A0A024R571',
 'A0A024R5G9',
 'A0A024R772',
 'A0A024R7P0',
 'A0A024R7W5',
 'A0A024R8F3',
 'A0A024RA87',
 'A0A024RBT8',
 'A0A024RCL3',
 'A0A024RCN4',
 'A0A024RDL5',
 'A0A044PY82',
 'A0A067XG54',
 'A0A067XG57',
 'A0A075B6E2',
 'A0A075B6E5',
 'A0A075B6E6',
 'A0A075B6E9',
 'A0A075B6F3',
 'A0A075B6F6',
 'A0A075B6F9',
 'A0A075B6G4',
 'A0A075B6G5',
 'A0A075B6G6',
 'A0A075B6G7',
 'A0A075B6G8',
 'A0A075B6H0',
 'A0A075B6H3',
 'A0A075B6H4',
 'A0A075B6P0',
 'A0A075B6P4',
 'A0A075B6P6',
 'A0A075B6P9',
 'A0A075B6Q6',
 'A0A075B6R4',
 'A0A075B6R5',
 'A0A075B6R8',
 'A0A075B6T1',
 'A0A075B6T9',
 'A0A075B723',
 'A0A075B724',
 'A0A075B727',
 'A0A075B728',
 'A0A075B729',
 'A0A075B730',
 'A0A075B731',
 'A0A075B735',
 'A0A075B736',
 'A0A075B737',
 'A0A075B740',
 'A0A075B743',
 'A0A075B748',
 'A0A075B749',
 'A0

In [ ]:
import os
import json
import gzip
from Bio import SeqIO
import pandas as pd

# ===== 경로 설정 =====
trembl_path = r"E:\CAGI_data\uniprot_trembl.fasta.gz"
json_path   = r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\UniProtID_to_seq.json"
input_dir   = r"E:\CAGI_data\All_Missense_Annotated"
chroms      = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY", "chrM"]

# ===== 기존 JSON 불러오기 =====
with open(json_path, "r") as f:
    id_to_seq = json.load(f)

print(f"현재 JSON에 포함된 UniProtID 개수: {len(id_to_seq)}")

# ===== 1. trembl fasta에서 unmatched_ids 검색 =====
target_ids = set(unmatched_ids)  # 이미 확보한 unmatched_ids
found = {}

with gzip.open(trembl_path, "rt") as handle:
    for record in SeqIO.parse(handle, "fasta"):
        uid = record.id.split("|")[1] if "|" in record.id else record.id.split()[0]
        if uid in target_ids:
            found[uid] = str(record.seq)
            print(f"✅ Found in trembl: {uid}")
        if len(found) == len(target_ids):
            break

# ===== 2. JSON 업데이트 =====
if found:
    id_to_seq.update(found)
    with open(json_path, "w") as f:
        json.dump(id_to_seq, f)
    print(f"\n📦 JSON 업데이트 완료: {len(found)}개 ID 추가됨")
else:
    print("\n⚠️ trembl에서 추가로 찾은 ID 없음")

# ===== 3. 최종 invalid_ids 확정 =====
invalid_ids = target_ids - found.keys()
print(f"\n❗ 최종 invalid_ids 개수: {len(invalid_ids)}")

# ===== 4. chr별 TSV에서 invalid_ids 제거 =====
for chrom in chroms:
    path = os.path.join(input_dir, f"missense_for_prediction_{chrom}.tsv")
    if not os.path.exists(path):
        print(f"⚠️ Missing file: {path}")
        continue

    df = pd.read_csv(path, sep="\t", dtype=str)
    before = df.shape[0]

    df = df[~df["UniProtID"].isin(invalid_ids)].reset_index(drop=True)
    after = df.shape[0]

    out_path = os.path.join(input_dir, f"missense_for_prediction_{chrom}_cleaned.tsv")
    df.to_csv(out_path, sep="\t", index=False)

    print(f"✅ {chrom}: {before} → {after} rows (filtered {before-after}), saved to {out_path}")


현재 JSON에 포함된 UniProtID 개수: 573664
✅ Found in trembl: H0YLB2
✅ Found in trembl: A0A6I8PLD9
✅ Found in trembl: A6NHU9
✅ Found in trembl: D2DJS5
✅ Found in trembl: Q7Z536
✅ Found in trembl: W4VSQ8
✅ Found in trembl: A0A087X090
✅ Found in trembl: A0A0S2Z471
✅ Found in trembl: A0A3B3ISR2
✅ Found in trembl: E5RI58
✅ Found in trembl: E9PS44
✅ Found in trembl: G3V336
✅ Found in trembl: H3BNV2
✅ Found in trembl: H3BUS4
✅ Found in trembl: H7C0W5
✅ Found in trembl: A0A2R8Y7Q5
✅ Found in trembl: B3KXV3
✅ Found in trembl: G3V427
✅ Found in trembl: Q6P2S0
✅ Found in trembl: B4DLJ1
✅ Found in trembl: K7EM56
✅ Found in trembl: A0A7I2V3E2
✅ Found in trembl: A0A8V8TPM0
✅ Found in trembl: B2RNG4
✅ Found in trembl: B4DFN3
✅ Found in trembl: B5MCR8
✅ Found in trembl: C9JLG1
✅ Found in trembl: F5H6F7
✅ Found in trembl: G5E9A6
✅ Found in trembl: H0Y459
✅ Found in trembl: Q5T8U5
✅ Found in trembl: Q6ZMD1
✅ Found in trembl: A2A274
✅ Found in trembl: B1B1F5
✅ Found in trembl: H3BSE0
✅ Found in trembl: M0R1D2
✅ 